# NSGABLACK工具集详解（二）：实用工具与并行计算

NSGABLACK框架提供了丰富的实用工具，提升优化效率和可扩展性：

## Utils模块实用工具

1. **并行评估器** - 多核加速评估
2. **内存管理器** - 智能内存优化
3. **实验管理器** - 实验组织与追踪
4. **数组工具** - 高效数组操作
5. **求解器扩展** - 增强求解器功能
6. **快速非支配排序** - 多目标优化核心算法

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Callable, Optional
from functools import partial
import psutil
import os
from collections import deque
import json
from datetime import datetime

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"CPU核心数: {mp.cpu_count()}")
print(f"可用内存: {psutil.virtual_memory().total / 1024**3:.1f} GB")

## 1. 并行评估器（Parallel Evaluator）

In [ ]:
class ParallelEvaluator:
    """并行评估器 - 支持多进程和多线程评估"""
    
    def __init__(self, n_workers: int = None, backend: str = 'multiprocessing', 
                 chunk_size: int = None, timeout: float = None):
        """
        初始化并行评估器
        
        Args:
            n_workers: 工作进程数，None表示使用所有核心
            backend: 并行后端 ('multiprocessing' 或 'threading')
            chunk_size: 任务块大小
            timeout: 单个任务超时时间（秒）
        """
        self.n_workers = n_workers or mp.cpu_count()
        self.backend = backend
        self.chunk_size = chunk_size
        self.timeout = timeout
        
        # 性能统计
        self.total_evaluations = 0
        self.total_time = 0
        self.failed_tasks = 0
        
        print(f"创建并行评估器")
        print(f"  工作进程数: {self.n_workers}")
        print(f"  后端: {backend}")
        if timeout:
            print(f"  超时时间: {timeout}秒")
    
    def evaluate(self, func: Callable, candidates: List[Any], 
                progress_callback: Callable = None) -> List[Any]:
        """
        并行评估候选解
        
        Args:
            func: 评估函数
            candidates: 候选解列表
            progress_callback: 进度回调函数
        
        Returns:
            评估结果列表
        """
        if not candidates:
            return []
        
        start_time = time.time()
        results = [None] * len(candidates)
        
        # 选择执行器
        executor_class = (ProcessPoolExecutor if self.backend == 'multiprocessing' 
                         else ThreadPoolExecutor)
        
        # 分块处理（如果需要）
        if self.chunk_size and len(candidates) > self.chunk_size:
            results = self._evaluate_chunked(func, candidates, executor_class, progress_callback)
        else:
            results = self._evaluate_direct(func, candidates, executor_class, progress_callback)
        
        # 更新统计
        self.total_evaluations += len(candidates)
        self.total_time += time.time() - start_time
        
        return results
    
    def _evaluate_direct(self, func: Callable, candidates: List[Any], 
                         executor_class, progress_callback) -> List[Any]:
        """直接评估"""
        results = [None] * len(candidates)
        
        with executor_class(max_workers=self.n_workers) as executor:
            # 提交任务
            future_to_index = {\n                executor.submit(func, candidate): i 
                for i, candidate in enumerate(candidates)
            }
            
            # 收集结果
            completed = 0
            for future in as_completed(future_to_index, timeout=self.timeout):
                index = future_to_index[future]
                try:
                    results[index] = future.result()
                    completed += 1
                    
                    # 进度回调
                    if progress_callback:
                        progress_callback(completed, len(candidates))
                        
                except Exception as e:
                    results[index] = None
                    self.failed_tasks += 1
                    print(f"评估失败 (索引{index}): {e}")
        
        return results
    
    def _evaluate_chunked(self, func: Callable, candidates: List[Any], 
                          executor_class, progress_callback) -> List[Any]:
        """分块评估"""
        results = []
        n_chunks = (len(candidates) + self.chunk_size - 1) // self.chunk_size
        
        def process_chunk(chunk):
            return [func(item) for item in chunk]
        
        with executor_class(max_workers=self.n_workers) as executor:
            # 分块
            chunks = [candidates[i*self.chunk_size:(i+1)*self.chunk_size] 
                     for i in range(n_chunks)]
            
            # 提交块任务
            future_to_chunk = {
                executor.submit(process_chunk, chunk): i 
                for i, chunk in enumerate(chunks)
            }
            
            # 收集结果
            chunk_results = [None] * n_chunks
            completed = 0
            
            for future in as_completed(future_to_chunk):
                chunk_index = future_to_chunk[future]
                try:
                    chunk_results[chunk_index] = future.result()
                    completed += 1
                    
                    if progress_callback:
                        progress = completed * self.chunk_size
                        progress_callback(min(progress, len(candidates)), len(candidates))
                        
                except Exception as e:
                    print(f"块评估失败 (块{chunk_index}): {e}")
                    # 填充None
                    chunk_size_actual = len(chunks[chunk_index])
                    chunk_results[chunk_index] = [None] * chunk_size_actual
            
            # 合并结果
            for chunk_result in chunk_results:
                if chunk_result:
                    results.extend(chunk_result)
        
        return results
    
    def evaluate_async(self, func: Callable, candidates: List[Any]):
        """异步评估生成器"""
        executor_class = (ProcessPoolExecutor if self.backend == 'multiprocessing' 
                         else ThreadPoolExecutor)
        
        with executor_class(max_workers=self.n_workers) as executor:
            # 提交所有任务
            futures = [executor.submit(func, candidate) for candidate in candidates]
            
            # 生成结果
            for future in as_completed(futures):
                try:
                    yield future.result()
                except Exception as e:
                    yield None
    
    def get_statistics(self) -> Dict[str, Any]:
        """获取性能统计"""
        return {
            'total_evaluations': self.total_evaluations,
            'total_time': self.total_time,
            'evaluations_per_second': self.total_evaluations / max(self.total_time, 0.001),
            'failed_tasks': self.failed_tasks,
            'failure_rate': self.failed_tasks / max(self.total_evaluations, 1),
            'n_workers': self.n_workers,
            'backend': self.backend
        }

# 测试函数
def expensive_function(x):
    """模拟昂贵的评估函数"""
    time.sleep(0.01)  # 模拟计算时间
    x = np.asarray(x)
    return np.sum(x**2) + np.sin(np.sum(x))

def multi_objective_function(x):
    """多目标评估函数"""
    time.sleep(0.005)
    x = np.asarray(x)
    f1 = np.sum(x**2)
    f2 = np.sum((x - 2)**2)
    return np.array([f1, f2])

# 测试并行评估器
print("测试并行评估器：")
print("=" * 50)

# 生成测试数据
n_candidates = 50
candidates = [np.random.randn(5) for _ in range(n_candidates)]

# 串行评估基准
print("\n1. 串行评估基准：")
start_time = time.time()
serial_results = [expensive_function(c) for c in candidates]
serial_time = time.time() - start_time
print(f"  评估{n_candidates}个候选解耗时: {serial_time:.2f}秒")

# 并行评估
print("\n2. 并行评估：")
evaluator = ParallelEvaluator(n_workers=4, backend='multiprocessing')
start_time = time.time()
parallel_results = evaluator.evaluate(expensive_function, candidates)
parallel_time = time.time() - start_time
print(f"  评估{n_candidates}个候选解耗时: {parallel_time:.2f}秒")
print(f"  加速比: {serial_time/parallel_time:.2f}x")

# 验证结果
print(f"\n3. 结果验证：")
print(f"  结果一致: {np.allclose(serial_results, parallel_results)}")

# 性能对比测试
print("\n4. 性能对比：")
sizes = [20, 50, 100, 200]
worker_counts = [1, 2, 4]

plt.figure(figsize=(12, 5))

# 时间对比
plt.subplot(1, 2, 1)
for n_workers in worker_counts:
    times = []
    for size in sizes:
        test_candidates = candidates[:size]
        evalutor = ParallelEvaluator(n_workers=n_workers)
        start = time.time()
        evalutor.evaluate(expensive_function, test_candidates)
        times.append(time.time() - start)
    plt.plot(sizes, times, 'o-', label=f'{n_workers}进程', linewidth=2)

plt.xlabel('候选解数量')
plt.ylabel('评估时间（秒）')
plt.title('并行评估性能')
plt.legend()
plt.grid(True, alpha=0.3)

# 加速比
plt.subplot(1, 2, 2)
speedups = []
for n_workers in worker_counts[1:]:  # 跳过1进程作为基准
    speedup = []
    for size in sizes:
        test_candidates = candidates[:size]
        # 单进程时间
        eval1 = ParallelEvaluator(n_workers=1)
        start = time.time()
        eval1.evaluate(expensive_function, test_candidates)
        time1 = time.time() - start
        
        # 多进程时间
        evaln = ParallelEvaluator(n_workers=n_workers)
        start = time.time()
        evaln.evaluate(expensive_function, test_candidates)
        timen = time.time() - start
        
        speedup.append(time1 / timen if timen > 0 else 0)
    
    speedups.append(speedup)
    plt.plot(sizes, speedup, 'o-', label=f'{n_workers}进程', linewidth=2)

# 理想加速比
ideal_speedup = [n_workers for n_workers in worker_counts[1:]]
for i, n_workers in enumerate(worker_counts[1:]):
    plt.axhline(y=n_workers, color=f'C{i}', linestyle='--', alpha=0.5)

plt.xlabel('候选解数量')
plt.ylabel('加速比')
plt.title('并行加速比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 获取统计信息
stats = evaluator.get_statistics()
print("\n5. 评估器统计：")
for key, value in stats.items():
    print(f"  {key}: {value:.2f}" if isinstance(value, float) else f"  {key}: {value}")

## 2. 内存管理器（Memory Manager）

In [ ]:
import weakref
from collections import OrderedDict

class MemoryManager:
    """智能内存管理器"""
    
    def __init__(self, max_memory_mb: int = 1024, 
                 cache_strategy: str = 'lru', 
                 gc_threshold: float = 0.8):
        """
        初始化内存管理器
        
        Args:
            max_memory_mb: 最大内存使用量（MB）
            cache_strategy: 缓存策略 ('lru', 'lfu', 'fifo')
            gc_threshold: 触发垃圾回收的内存阈值（比例）
        """
        self.max_memory_bytes = max_memory_mb * 1024 * 1024
        self.cache_strategy = cache_strategy
        self.gc_threshold = gc_threshold
        
        # 缓存存储
        self.cache = OrderedDict() if cache_strategy == 'fifo' else {}
        self.access_count = {}  # LFU使用
        self.access_time = {}  # LRU使用
        
        # 内存统计
        self.allocated_memory = 0
        self.cache_hits = 0
        self.cache_misses = 0
        self.evictions = 0
        
        print(f"创建内存管理器")
        print(f"  最大内存: {max_memory_mb} MB")
        print(f"  缓存策略: {cache_strategy}")
    
    def cache_data(self, key: str, data: Any, priority: int = 1, 
                   estimated_size: int = None) -> bool:
        """
        缓存数据
        
        Args:
            key: 数据键
            data: 数据对象
            priority: 优先级（1-5，5最高）
            estimated_size: 估算大小（字节），None表示自动估算
        
        Returns:
            是否成功缓存
        """
        # 估算数据大小
        if estimated_size is None:
            estimated_size = self._estimate_size(data)
        
        # 检查是否需要清理缓存
        while self.allocated_memory + estimated_size > self.max_memory_bytes and self.cache:
            self._evict_item()
        
        # 如果仍然放不下，拒绝缓存
        if self.allocated_memory + estimated_size > self.max_memory_bytes:
            return False
        
        # 移除现有项（如果存在）
        if key in self.cache:
            self._remove_item(key)
        
        # 添加新项
        cache_item = {
            'data': data,
            'size': estimated_size,
            'priority': priority,
            'timestamp': time.time()
        }
        
        if self.cache_strategy == 'fifo':
            self.cache[key] = cache_item
        else:
            self.cache[key] = cache_item
            self.access_count[key] = 1
            self.access_time[key] = time.time()
        
        self.allocated_memory += estimated_size
        return True
    
    def get_cached_data(self, key: str) -> Any:
        """获取缓存数据"""
        if key not in self.cache:
            self.cache_misses += 1
            return None
        
        # 更新访问统计
        self.cache_hits += 1
        
        if self.cache_strategy == 'lru':
            self.access_time[key] = time.time()
        elif self.cache_strategy == 'lfu':
            self.access_count[key] += 1
        
        # FIFO策略需要重新插入末尾
        if self.cache_strategy == 'fifo':
            item = self.cache.pop(key)
            self.cache[key] = item
        
        return self.cache[key]['data']
    
    def _evict_item(self):
        """根据策略驱逐项"""
        if not self.cache:
            return
        
        if self.cache_strategy == 'fifo':
            # FIFO: 移除最早插入的
            key = next(iter(self.cache))
        elif self.cache_strategy == 'lru':
            # LRU: 移除最久未访问的
            key = min(self.access_time.keys(), key=lambda k: self.access_time[k])
        elif self.cache_strategy == 'lfu':
            # LFU: 移除访问频率最低的
            key = min(self.access_count.keys(), key=lambda k: self.access_count[k])
        else:
            # 默认随机
            key = next(iter(self.cache))
        
        self._remove_item(key)
        self.evictions += 1
    
    def _remove_item(self, key: str):
        """移除缓存项"""
        if key in self.cache:
            self.allocated_memory -= self.cache[key]['size']
            del self.cache[key]
            
            if key in self.access_count:
                del self.access_count[key]
            if key in self.access_time:
                del self.access_time[key]
    
    def _estimate_size(self, obj: Any) -> int:
        """估算对象大小（字节）"""
        if isinstance(obj, np.ndarray):
            return obj.nbytes
        elif isinstance(obj, (list, tuple)):
            return sum(self._estimate_size(item) for item in obj)
        elif isinstance(obj, dict):
            return sum(self._estimate_size(v) for v in obj.values())
        elif isinstance(obj, str):
            return len(obj.encode('utf-8'))
        elif isinstance(obj, (int, float)):
            return 8
        else:
            # 默认估算
            return 100
    
    def get_memory_info(self) -> Dict[str, Any]:
        """获取内存使用信息"""
        process = psutil.Process()
        memory_info = process.memory_info()
        
        return {
            'process_rss_mb': memory_info.rss / 1024 / 1024,
            'process_vms_mb': memory_info.vms / 1024 / 1024,
            'system_available_mb': psutil.virtual_memory().available / 1024 / 1024,
            'cache_allocated_mb': self.allocated_memory / 1024 / 1024,
            'cache_items': len(self.cache),
            'cache_utilization': self.allocated_memory / self.max_memory_bytes,
            'hit_rate': self.cache_hits / max(self.cache_hits + self.cache_misses, 1)
        }
    
    def clear_cache(self):
        """清空缓存"""
        self.cache.clear()
        self.access_count.clear()
        self.access_time.clear()
        self.allocated_memory = 0
        print(f"缓存已清空")
    
    def optimize_cache(self):
        """优化缓存（垃圾回收）"""
        import gc
        
        # 收集弱引用
        weak_refs = []
        for key, item in list(self.cache.items()):
            if 'data' in item and item['data'] is not None:
                try:
                    weak_ref = weakref.ref(item['data'])
                    if weak_ref() is None:
                        # 对象已被回收
                        self._remove_item(key)
                except:
                    pass
        
        # 强制垃圾回收
        gc.collect()
        
        print(f"缓存优化完成")

# 测试内存管理器
print("\n测试内存管理器：")
print("=" * 50)

# 创建不同策略的内存管理器
managers = {
    'LRU': MemoryManager(max_memory_mb=10, cache_strategy='lru'),
    'LFU': MemoryManager(max_memory_mb=10, cache_strategy='lfu'),
    'FIFO': MemoryManager(max_memory_mb=10, cache_strategy='fifo')
}

# 测试缓存性能
print("\n1. 缓存策略对比：")
test_data = [np.random.rand(100, 100) for _ in range(30)]  # 约30MB数据

for strategy, manager in managers.items():
    print(f"\n{strategy}策略：")
    
    # 缓存数据
    cached_count = 0
    for i, data in enumerate(test_data):
        if manager.cache_data(f"data_{i}", data, priority=i % 3 + 1):
            cached_count += 1
    
    # 访问数据（模拟不同访问模式）
    if strategy == 'LRU':
        # 顺序访问
        for i in range(20):
            manager.get_cached_data(f"data_{i}")
    elif strategy == 'LFU':
        # 频繁访问某些项
        for _ in range(5):
            for i in range(5):
                manager.get_cached_data(f"data_{i}")
    
    stats = manager.get_memory_info()
    print(f"  缓存项数: {cached_count}")
    print(f"  内存使用: {stats['cache_allocated_mb']:.2f} MB")
    print(f"  命中率: {stats['hit_rate']:.2%}")
    print(f"  驱逐次数: {manager.evictions}")

# 内存使用监控
print("\n\n2. 内存使用监控：")
monitor_manager = MemoryManager(max_memory_mb=50, cache_strategy='lru')

# 动态添加数据
memory_history = []
cache_size_history = []
time_points = []

for i in range(20):
    # 添加数据
    data = np.random.rand(200, 200 * (i % 5 + 1))  # 大小变化
    monitor_manager.cache_data(f"dynamic_{i}", data)
    
    # 记录状态
    info = monitor_manager.get_memory_info()
    memory_history.append(info['process_rss_mb'])
    cache_size_history.append(info['cache_allocated_mb'])
    time_points.append(i)
    
    # 随机访问
    if np.random.random() < 0.3:
        key = f"dynamic_{np.random.randint(0, i+1)}"
        monitor_manager.get_cached_data(key)
    
    time.sleep(0.1)

# 可视化
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time_points, memory_history, 'b-', label='进程内存', linewidth=2)
plt.plot(time_points, cache_size_history, 'r-', label='缓存内存', linewidth=2)
plt.xlabel('时间点')
plt.ylabel('内存使用 (MB)')
plt.title('内存使用变化')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
final_stats = monitor_manager.get_memory_info()
metrics = ['进程内存', '缓存内存', '命中率']
values = [
    final_stats['process_rss_mb'],
    final_stats['cache_allocated_mb'],
    final_stats['hit_rate'] * 100
]
colors = ['blue', 'red', 'green']
bars = plt.bar(metrics, values, alpha=0.7, color=colors)
plt.ylabel('值')
plt.title('最终状态')
plt.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{val:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 最终统计
print("\n3. 最终内存统计：")
for key, value in final_stats.items():
    if 'rate' in key:
        print(f"  {key}: {value:.2%}")
    else:
        print(f"  {key}: {value:.2f}")

## 3. 实验管理器（Experiment Manager）

In [ ]:
class ExperimentManager:
    """实验管理器 - 组织和管理优化实验"""
    
    def __init__(self, experiment_name: str, save_dir: str = "experiments",
                 auto_save: bool = True):
        """
        初始化实验管理器
        
        Args:
            experiment_name: 实验名称
            save_dir: 保存目录
            auto_save: 是否自动保存
        """
        self.experiment_name = experiment_name
        self.save_dir = save_dir
        self.auto_save = auto_save
        
        # 创建实验目录
        self.experiment_dir = os.path.join(save_dir, experiment_name)
        os.makedirs(self.experiment_dir, exist_ok=True)
        
        # 实验数据
        self.start_time = datetime.now()
        self.runs = {}
        self.config = {}
        self.global_results = {}
        self.metadata = {
            'created_at': self.start_time.isoformat(),
            'version': '1.0',
            'tags': []
            'description': ''
        }
        
        print(f"创建实验管理器")
        print(f"  实验名称: {experiment_name}")
        print(f"  保存目录: {self.experiment_dir}")
    
    def add_run(self, run_name: str, solver_name: str, problem_name: str,
                config: Dict[str, Any] = None) -> str:
        """
        添加实验运行
        
        Args:
            run_name: 运行名称
            solver_name: 求解器名称
            problem_name: 问题名称
            config: 配置参数
        
        Returns:
            运行ID
        """
        run_id = f"run_{len(self.runs)}_{int(time.time())}"
        
        self.runs[run_id] = {
            'run_id': run_id,
            'run_name': run_name,
            'solver_name': solver_name,
            'problem_name': problem_name,
            'config': config or {},
            'status': 'pending',
            'created_at': datetime.now().isoformat(),
            'results': {},
            'logs': []
        }
        
        if self.auto_save:
            self._save_experiment()
        
        return run_id
    
    def start_run(self, run_id: str):
        """开始运行"""
        if run_id in self.runs:
            self.runs[run_id]['status'] = 'running'
            self.runs[run_id]['started_at'] = datetime.now().isoformat()
            self.runs[run_id]['logs'].append(f"运行开始: {datetime.now()}")
    
    def complete_run(self, run_id: str, results: Dict[str, Any]):
        """完成运行"""
        if run_id in self.runs:
            self.runs[run_id]['status'] = 'completed'
            self.runs[run_id]['completed_at'] = datetime.now().isoformat()
            self.runs[run_id]['results'] = results
            self.runs[run_id]['logs'].append(f"运行完成: {datetime.now()}")
            
            if self.auto_save:
                self._save_experiment()
    
    def fail_run(self, run_id: str, error: str):
        """运行失败"""
        if run_id in self.runs:
            self.runs[run_id]['status'] = 'failed'
            self.runs[run_id]['failed_at'] = datetime.now().isoformat()
            self.runs[run_id]['error'] = error
            self.runs[run_id]['logs'].append(f"运行失败: {datetime.now()} - {error}")
    
    def add_log(self, run_id: str, message: str):
        """添加日志"""
        if run_id in self.runs:
            self.runs[run_id]['logs'].append(f"{datetime.now()}: {message}")
    
    def compare_runs(self, run_ids: List[str] = None) -> Dict[str, Any]:
        """
        比较多个运行
        
        Args:
            run_ids: 要比较的运行ID，None表示所有完成的运行
        
        Returns:
            比较结果
        """
        if run_ids is None:
            run_ids = [rid for rid, run in self.runs.items() 
                      if run['status'] == 'completed']
        
        comparison = {
            'run_count': len(run_ids),
            'runs': {},
            'best_fitness': float('inf'),
            'worst_fitness': 0,
            'convergence_data': {}
        }
        
        all_fitness = []
        
        for run_id in run_ids:
            if run_id in self.runs:
                run = self.runs[run_id]
                results = run.get('results', {})
                
                comparison['runs'][run_id] = {
                    'name': run['run_name'],
                    'solver': run['solver_name'],
                    'best_fitness': results.get('best_fitness', float('inf')),
                    'n_evaluations': results.get('n_evaluations', 0),
                    'elapsed_time': results.get('elapsed_time', 0)
                }
                
                fitness = results.get('best_fitness', float('inf'))
                all_fitness.append(fitness)
                
                if fitness < comparison['best_fitness']:
                    comparison['best_fitness'] = fitness
                    comparison['best_run_id'] = run_id
                
                if fitness > comparison['worst_fitness']:
                    comparison['worst_fitness'] = fitness
                
                # 收敛数据
                if 'history' in results:
                    comparison['convergence_data'][run_id] = results['history']
        
        if all_fitness:
            comparison['mean_fitness'] = np.mean(all_fitness)
            comparison['std_fitness'] = np.std(all_fitness)
        
        return comparison
    
    def generate_report(self, run_ids: List[str] = None) -> str:
        """
        生成实验报告
        
        Args:
            run_ids: 要包含的运行ID
        
        Returns:
            报告文本
        """
        comparison = self.compare_runs(run_ids)
        
        report = f"# 实验报告: {self.experiment_name}\n\n"
        report += f"创建时间: {self.start_time}\n\n"
        
        report += f"## 实验概览\n"
        report += f"- 总运行数: {len(self.runs)}\n"
        report += f"- 完成运行数: {comparison['run_count']}\n"
        report += f"- 运行时间: {datetime.now() - self.start_time}\n\n"
        
        report += f"## 结果统计\n"
        report += f"- 最优适应度: {comparison['best_fitness']:.6f}\n"
        report += f"- 平均适应度: {comparison.get('mean_fitness', 0):.6f}\n"
        report += f"- 标准差: {comparison.get('std_fitness', 0):.6f}\n\n"
        
        report += f"## 运行详情\n"
        for run_id, run_info in comparison['runs'].items():
            report += f"### {run_info['name']}\n"
            report += f"- 求解器: {run_info['solver']}\n"
            report += f"- 最优适应度: {run_info['best_fitness']:.6f}\n"
            report += f"- 评估次数: {run_info['n_evaluations']}\n"
            report += f"- 运行时间: {run_info['elapsed_time']:.2f}秒\n\n"
        
        return report
    
    def visualize_results(self, run_ids: List[str] = None, save_plots: bool = True):
        """可视化结果"""
        comparison = self.compare_runs(run_ids)
        
        plt.figure(figsize=(15, 10))
        
        # 性能对比
        plt.subplot(2, 3, 1)
        names = [run['name'] for run in comparison['runs'].values()]
        fitnesses = [run['best_fitness'] for run in comparison['runs'].values()]
        bars = plt.bar(names, fitnesses, alpha=0.7)
        plt.ylabel('最优适应度')
        plt.title('最优适应度对比')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3, axis='y')
        
        # 评估次数对比
        plt.subplot(2, 3, 2)
        evaluations = [run['n_evaluations'] for run in comparison['runs'].values()]
        bars = plt.bar(names, evaluations, alpha=0.7, color='orange')
        plt.ylabel('评估次数')
        plt.title('评估次数对比')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3, axis='y')
        
        # 运行时间对比
        plt.subplot(2, 3, 3)
        times = [run['elapsed_time'] for run in comparison['runs'].values()]
        bars = plt.bar(names, times, alpha=0.7, color='green')
        plt.ylabel('运行时间 (秒)')
        plt.title('运行时间对比')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3, axis='y')
        
        # 收敛曲线
        if comparison['convergence_data']:
            plt.subplot(2, 3, 4)
            for run_id, history in comparison['convergence_data'].items():
                run_name = comparison['runs'][run_id]['name']
                plt.plot(history, label=run_name, linewidth=2)
            plt.xlabel('迭代')
            plt.ylabel('适应度')
            plt.title('收敛曲线')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.yscale('log')
        
        # 效率散点图
        plt.subplot(2, 3, 5)
        plt.scatter(evaluations, fitnesses, s=100, alpha=0.7)
        for i, name in enumerate(names):
            plt.annotate(name, (evaluations[i], fitnesses[i]), 
                        xytext=(5, 5), textcoords='offset points')
        plt.xlabel('评估次数')
        plt.ylabel('最优适应度')
        plt.title('效率分析')
        plt.grid(True, alpha=0.3)
        
        # 状态饼图
        plt.subplot(2, 3, 6)
        status_counts = {}
        for run in self.runs.values():
            status = run['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        labels = list(status_counts.keys())
        sizes = list(status_counts.values())
        colors = ['green', 'red', 'yellow', 'gray']
        
        plt.pie(sizes, labels=labels, colors=colors[:len(labels)], 
               autopct='%1.1f%%', startangle=90)
        plt.title('运行状态分布')
        
        plt.tight_layout()
        
        if save_plots:
            plot_path = os.path.join(self.experiment_dir, 'results_visualization.png')
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            print(f"可视化已保存到: {plot_path}")
        
        plt.show()
    
    def _save_experiment(self):
        """保存实验数据"""
        experiment_data = {
            'experiment_name': self.experiment_name,
            'metadata': self.metadata,
            'config': self.config,
            'runs': self.runs,
            'global_results': self.global_results
        }
        
        save_path = os.path.join(self.experiment_dir, 'experiment.json')
        with open(save_path, 'w') as f:
            json.dump(experiment_data, f, indent=2)
    
    def load_experiment(self, filepath: str = None):
        """加载实验数据"""
        if filepath is None:
            filepath = os.path.join(self.experiment_dir, 'experiment.json')
        
        with open(filepath, 'r') as f:
            experiment_data = json.load(f)
        
        self.experiment_name = experiment_data['experiment_name']
        self.metadata = experiment_data['metadata']
        self.config = experiment_data['config']
        self.runs = experiment_data['runs']
        self.global_results = experiment_data['global_results']

# 测试实验管理器
print("\n测试实验管理器：")
print("=" * 50)

# 创建实验管理器
exp_manager = ExperimentManager("optimization_comparison_2024")

# 添加多个运行
configs = [
    {'population_size': 50, 'mutation_rate': 0.1},
    {'population_size': 100, 'mutation_rate': 0.05},
    {'population_size': 200, 'mutation_rate': 0.02}
]

run_ids = []
for i, config in enumerate(configs):
    run_id = exp_manager.add_run(
        run_name=f"GA_run_{i+1}",
        solver_name="GeneticAlgorithm",
        problem_name="Sphere",
        config=config
    )
    run_ids.append(run_id)

# 模拟运行
print("\n模拟实验运行...")
for i, run_id in enumerate(run_ids):
    exp_manager.start_run(run_id)
    exp_manager.add_log(run_id, f"开始运行，配置: {configs[i]}")
    
    # 模拟优化结果
    time.sleep(0.5)  # 模拟运行时间
    
    results = {
        'best_fitness': 0.1 * (i + 1) + np.random.normal(0, 0.01),
        'best_solution': np.random.randn(5) * 0.1,
        'n_evaluations': 1000 * (i + 1),
        'elapsed_time': 0.5 + i * 0.2,
        'history': list(np.random.exponential(1, 50).cumsum())
    }
    
    if i == 1:
        # 模拟一个失败的运行
        exp_manager.fail_run(run_id, "模拟错误: 收敛失败")
    else:
        exp_manager.complete_run(run_id, results)
    
    print(f"  运行 {i+1} 完成")

# 生成报告
report = exp_manager.generate_report()
print("\n实验报告：")
print(report[:500] + "..." if len(report) > 500 else report)

# 可视化结果
print("\n生成可视化...")
exp_manager.visualize_results(save_plots=False)

## 4. 数组工具（Array Utils）

In [ ]:
class ArrayUtils:
    """数组实用工具集"""
    
    @staticmethod
    def ensure_2d(array: np.ndarray) -> np.ndarray:
        """确保数组是2维的"""
        array = np.asarray(array)
        if array.ndim == 1:
            return array.reshape(1, -1)
        elif array.ndim > 2:
            return array.reshape(array.shape[0], -1)
        return array
    
    @staticmethod
    def normalize(array: np.ndarray, method: str = 'minmax', 
                  axis: Optional[int] = None, 
                  target_range: Tuple[float, float] = (0, 1)) -> np.ndarray:
        """
        数组归一化
        
        Args:
            array: 输入数组
            method: 归一化方法 ('minmax', 'zscore', 'robust', 'unit')
            axis: 归一化轴
            target_range: 目标范围（仅minmax）
        """
        array = np.asarray(array)
        
        if method == 'minmax':
            min_val = np.min(array, axis=axis, keepdims=True)
            max_val = np.max(array, axis=axis, keepdims=True)
            range_val = max_val - min_val
            range_val = np.where(range_val == 0, 1, range_val)  # 避免除零
            normalized = (array - min_val) / range_val
            # 缩放到目标范围
            normalized = normalized * (target_range[1] - target_range[0]) + target_range[0]
            
        elif method == 'zscore':
            mean_val = np.mean(array, axis=axis, keepdims=True)
            std_val = np.std(array, axis=axis, keepdims=True)
            std_val = np.where(std_val == 0, 1, std_val)
            normalized = (array - mean_val) / std_val
            
        elif method == 'robust':
            median_val = np.median(array, axis=axis, keepdims=True)
            mad_val = np.median(np.abs(array - median_val), axis=axis, keepdims=True)
            mad_val = np.where(mad_val == 0, 1, mad_val)
            normalized = (array - median_val) / mad_val
            
        elif method == 'unit':
            norm_val = np.linalg.norm(array, axis=axis, keepdims=True)
            norm_val = np.where(norm_val == 0, 1, norm_val)
            normalized = array / norm_val
            
        else:
            raise ValueError(f"未知的归一化方法: {method}")
        
        return normalized
    
    @staticmethod
    def crowding_distance(objectives: np.ndarray) -> np.ndarray:
        """
        计算拥挤度距离
        
        Args:
            objectives: 目标值数组 (n_points, n_objectives)
        
        Returns:
            拥挤度距离数组
        """
        objectives = np.asarray(objectives)
        if objectives.ndim == 1:
            objectives = objectives.reshape(-1, 1)
        
        n_points, n_objectives = objectives.shape
        crowding = np.zeros(n_points)
        
        if n_points == 0:
            return crowding
        
        # 边界点设为无穷大
        if n_points > 0:
            crowding[0] = crowding[-1] = np.inf
        
        # 对每个目标计算拥挤度
        for i in range(n_objectives):
            # 按第i个目标排序
            sorted_indices = np.argsort(objectives[:, i])
            
            # 目标值范围
            obj_range = objectives[sorted_indices[-1], i] - objectives[sorted_indices[0], i]
            
            if obj_range > 0:
                # 计算中间点的拥挤度
                for j in range(1, n_points - 1):
                    crowding[sorted_indices[j]] += (
                        objectives[sorted_indices[j+1], i] - objectives[sorted_indices[j-1], i]
                    ) / obj_range
        
        return crowding
    
    @staticmethod
    def pareto_dominance(obj1: np.ndarray, obj2: np.ndarray) -> bool:
        """
        检查obj1是否支配obj2
        
        Args:
            obj1: 第一个目标向量
            obj2: 第二个目标向量
        
        Returns:
            obj1是否支配obj2
        """
        obj1 = np.asarray(obj1)
        obj2 = np.asarray(obj2)
        
        return np.all(obj1 <= obj2) and np.any(obj1 < obj2)
    
    @staticmethod
    def distance_matrix(points1: np.ndarray, points2: np.ndarray = None, 
                       metric: str = 'euclidean', **kwargs) -> np.ndarray:
        """
        计算距离矩阵
        
        Args:
            points1: 第一个点集 (n1, d)
            points2: 第二个点集 (n2, d)，None表示使用points1
            metric: 距离度量
            **kwargs: 额外参数
        
        Returns:
            距离矩阵 (n1, n2)
        """
        points1 = np.asarray(points1)
        if points2 is None:
            points2 = points1
        else:
            points2 = np.asarray(points2)
        
        if metric == 'euclidean':
            # 使用广播计算所有点对距离
            diff = points1[:, np.newaxis, :] - points2[np.newaxis, :, :]
            distances = np.sqrt(np.sum(diff**2, axis=2))
        elif metric == 'manhattan':
            diff = points1[:, np.newaxis, :] - points2[np.newaxis, :, :]
            distances = np.sum(np.abs(diff), axis=2)
        elif metric == 'chebyshev':
            diff = points1[:, np.newaxis, :] - points2[np.newaxis, :, :]
            distances = np.max(np.abs(diff), axis=2)
        elif metric == 'minkowski':
            p = kwargs.get('p', 2)
            diff = points1[:, np.newaxis, :] - points2[np.newaxis, :, :]
            distances = np.power(np.sum(np.power(np.abs(diff), p), axis=2), 1/p)
        elif metric == 'cosine':
            # 余弦相似度 = 1 - 余弦距离
            norm1 = np.linalg.norm(points1, axis=1, keepdims=True)
            norm2 = np.linalg.norm(points2, axis=1, keepdims=True)
            norm1 = np.where(norm1 == 0, 1, norm1)
            norm2 = np.where(norm2 == 0, 1, norm2)
            cosine_sim = np.dot(points1 / norm1, (points2 / norm2).T)
            distances = 1 - cosine_sim
        else:
            raise ValueError(f"未知的距离度量: {metric}")
        
        return distances
    
    @staticmethod
    def diversity_indicator(solutions: np.ndarray, k: int = 5) -> float:
        """
        计算解集的多样性指标
        
        Args:
            solutions: 解集 (n_solutions, n_variables)
            k: 最近邻数量
        
        Returns:
            多样性指标
        """
        solutions = np.asarray(solutions)
        n = len(solutions)
        
        if n <= 1:
            return 0.0
        
        # 计算距离矩阵
        distances = ArrayUtils.distance_matrix(solutions)
        
        # 排序，排除自身（距离为0）
        sorted_distances = np.sort(distances, axis=1)
        k_neighbors = sorted_distances[:, 1:k+1]  # 排除自己
        
        # 计算平均最近邻距离
        avg_distances = np.mean(k_neighbors, axis=1)
        diversity = np.mean(avg_distances)
        
        return diversity
    
    @staticmethod
    def hypervolume_indicator(points: np.ndarray, reference_point: np.ndarray) -> float:
        """
        计算超体积指标（仅适用于2D和3D）
        
        Args:
            points: 帕累托前沿点 (n_points, n_objectives)
            reference_point: 参考点
        
        Returns:
            超体积值
        """
        points = np.asarray(points)
        reference_point = np.asarray(reference_point)
        
        n_points, n_objectives = points.shape
        
        if n_objectives == 2:
            # 2D情况
            volume = 0
            
            # 按第一个目标排序
            sorted_points = points[points[:, 0].argsort()]
            
            for point in sorted_points:
                if np.all(point <= reference_point):
                    width = reference_point[0] - point[0]
                    height = reference_point[1] - point[1]
                    volume += width * height
            
            return volume
        
        elif n_objectives == 3:
            # 3D情况（简化实现）
            volume = 0
            
            # 按第一个目标排序
            sorted_points = points[points[:, 0].argsort()]
            
            for point in sorted_points:
                if np.all(point <= reference_point):
                    width = reference_point[0] - point[0]
                    depth = reference_point[1] - point[1]
                    height = reference_point[2] - point[2]
                    volume += width * depth * height
            
            return volume
        
        else:
            raise ValueError(f"不支持{n_objectives}D超体积计算")

# 测试数组工具
print("\n测试数组工具：")
print("=" * 50)

# 创建测试数据
test_array = np.array([1, 2, 3, 4, 5])
test_matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
test_3d = np.random.randn(2, 3, 4)

# 维度处理
print("\n1. 维度处理：")
print(f"原始一维数组形状: {test_array.shape}")
print(f"转换为二维后: {ArrayUtils.ensure_2d(test_array).shape}")
print(f"原始3D数组形状: {test_3d.shape}")
print(f"转换为二维后: {ArrayUtils.ensure_2d(test_3d).shape}")

# 归一化
print("\n2. 归一化方法：")
data = np.random.randn(100, 3) * 10 + 5

methods = ['minmax', 'zscore', 'robust', 'unit']
for method in methods:
    normalized = ArrayUtils.normalize(data, method=method)
    print(f"\n{method.capitalize()}归一化:")
    print(f"  均值: {np.mean(normalized):.4f}")
    print(f"  标准差: {np.std(normalized):.4f}")
    print(f"  范围: [{np.min(normalized):.4f}, {np.max(normalized):.4f}]")

# 拥挤度距离
print("\n3. 拥挤度距离计算：")
objectives = np.array([
    [1, 5], [2, 4], [3, 3], [4, 2], [5, 1],
    [1.5, 4.5], [2.5, 3.5], [3.5, 2.5], [4.5, 1.5]
])
crowding = ArrayUtils.crowding_distance(objectives)
print(f"目标值: {objectives}")
print(f"拥挤度距离: {crowding}")

# 距离矩阵
print("\n4. 距离矩阵：")
points = np.array([[0, 0], [1, 1], [2, 0], [0, 2]])
metrics = ['euclidean', 'manhattan', 'chebyshev', 'cosine']

for metric in metrics:
    dist_matrix = ArrayUtils.distance_matrix(points, metric=metric)
    print(f"\n{metric.capitalize()}距离矩阵:")
    print(dist_matrix)

# 多样性和超体积
print("\n5. 多样性和超体积指标：")
pareto_front = np.array([
    [1, 5], [2, 3], [3, 2], [4, 1.5], [5, 1]
])

diversity = ArrayUtils.diversity_indicator(pareto_front)
hypervolume = ArrayUtils.hypervolume_indicator(pareto_front, np.array([6, 6]))

print(f"帕累托前沿: {pareto_front}")
print(f"多样性指标: {diversity:.4f}")
print(f"超体积指标: {hypervolume:.4f}")

# 可视化
plt.figure(figsize=(15, 10))

# 归一化对比
plt.subplot(2, 3, 1)
original_data = data[:, 0]  # 只显示第一维
for i, method in enumerate(methods[:3]):
    normalized = ArrayUtils.normalize(data, method=method)
    plt.scatter(normalized[:, 0], normalized[:, 1], 
               alpha=0.6, label=method, s=30)
plt.xlabel('特征1')
plt.ylabel('特征2')
plt.title('不同归一化方法对比')
plt.legend()
plt.grid(True, alpha=0.3)

# 拥挤度可视化
plt.subplot(2, 3, 2)
plt.scatter(objectives[:, 0], objectives[:, 1], 
           c=crowding, cmap='viridis', s=100)
plt.colorbar(label='拥挤度')
plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title('拥挤度分布')
plt.grid(True, alpha=0.3)

# 距离矩阵热图
plt.subplot(2, 3, 3)
dist_matrix = ArrayUtils.distance_matrix(points, metric='euclidean')
plt.imshow(dist_matrix, cmap='hot', interpolation='nearest')
plt.colorbar(label='距离')
plt.xlabel('点索引')
plt.ylabel('点索引')
plt.title('欧几里得距离矩阵')
plt.xticks(range(len(points)))
plt.yticks(range(len(points)))

# 帕累托前沿和超体积
plt.subplot(2, 3, 4)
plt.scatter(pareto_front[:, 0], pareto_front[:, 1], 
           s=100, c='red', label='帕累托前沿')
# 绘制超体积区域
x_fill = np.concatenate([pareto_front[:, 0], [6, 0]])
y_fill = np.concatenate([pareto_front[:, 1], [0, 6]])
plt.fill(x_fill, y_fill, alpha=0.3, color='blue', label='超体积区域')
plt.scatter(6, 6, s=200, c='green', marker='*', label='参考点')
plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title(f'超体积 = {hypervolume:.2f}')
plt.legend()
plt.grid(True, alpha=0.3)

# 支配关系
plt.subplot(2, 3, 5)
all_points = np.array([[2, 2], [3, 1], [1, 4], [2, 3]])
colors = ['blue', 'red', 'gray', 'gray']
labels = ['支配者', '被支配', '中性', '中性']
for i, point in enumerate(all_points):
    dominated = False
    for j, other in enumerate(all_points):
        if i != j and ArrayUtils.pareto_dominance(other, point):
            dominated = True
            break
    if dominated:
        colors[i] = 'red'
        labels[i] = '被支配'
    elif not dominated:
        colors[i] = 'blue'
        labels[i] = '非支配'
    
    plt.scatter(point[0], point[1], s=100, c=colors[i], label=labels[i])
    plt.text(point[0]+0.1, point[1]+0.1, f'P{i+1}')

plt.xlabel('目标1')
plt.ylabel('目标2')
plt.title('帕累托支配关系')
plt.legend()
plt.grid(True, alpha=0.3)

# 多样性指标变化
plt.subplot(2, 3, 6)
# 生成不同分布的点集
diversities = []
for spread in [0.1, 0.5, 1, 2, 5]:
    test_points = np.random.randn(20, 2) * spread
    div = ArrayUtils.diversity_indicator(test_points)
    diversities.append(div)

plt.bar(range(len(diversities)), diversities, 
       alpha=0.7, color='green')
plt.xlabel('分布程度')
plt.ylabel('多样性指标')
plt.title('不同分布的多样性')
plt.xticks(range(len(diversities)), ['紧密', '较密', '中等', '分散', '很分散'])
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 总结

本节介绍了NSGABLACK框架的实用工具集：

### 1. 并行评估器
- 多进程/多线程支持
- 灵活的任务分配策略
- 超时和错误处理
- 性能监控和统计

### 2. 内存管理器
- 多种缓存策略（LRU/LFU/FIFO）
- 智能内存监控
- 自动垃圾回收
- 缓存优化

### 3. 实验管理器
- 完整的实验生命周期管理
- 运行对比和分析
- 自动报告生成
- 结果可视化

### 4. 数组工具
- 高效的数组操作
- 多种归一化方法
- 拥挤度计算
- 距离矩阵计算
- 多样性和超体积指标

这些工具大大提升了NSGABLACK框架的易用性和性能，让用户能够更专注于算法设计而非底层实现。